In [ ]:
import ollama
import chromadb
import json
import hashlib

db_client = chromadb.PersistentClient(path='./db_note_data')
llm_client = ollama.Client()

In [ ]:
prompt = input("LLM DIRETO: O que deseja perguntar?")
messages_input = [
    {
        'role': 'system', 
        'content': f"You are a helpful assistant, answering user's questions"
    },
    {
        'role': 'system', 
        'content': f"Answer in portuguese"
    },
    {
        'role': 'user', 
        'content': prompt
    }
]

In [ ]:
answer = llm_client.chat('gemma2:2b', messages_input)
print(json.dumps(answer, indent=4, ensure_ascii=False))

In [ ]:
stream = llm_client.chat('gemma2:2b', messages_input, stream=True)

for chunk in stream:
    print(chunk["message"]["content"], end="")

In [ ]:
faqs = [
    {
        "question": "Quem está apresentando agora?",
        "answer": "Gustavo Velozo"
    },
    {
        "question": "Onde as pessoas presentes trabalham?",
        "answer": "ZUP e Itaú"
    },
    {
        "question": "O que tem no escritório da ZUP?",
        "answer": "Na parede do escritório tem a palavra UNAGI, em referência a série Friends. Além de uma mesa de sinuca, salas com referências a linguagens de programação e muito mais"
    },
    {
        "question": "O que é StackSpot?",
        "answer": "StackSpot é uma plataforma que permite criar, compartilhar e usar plugins, stacks, starters e actions para acelerar o desenvolvimento de software."
    },
    {
        "question": "Como posso criar um plugin no StackSpot?",
        "answer": "Para criar um plugin no StackSpot, você precisa seguir o template de documentação fornecido e preencher as informações necessárias. Depois, você pode usar o comando `stk apply plugin` para aplicar o plugin na sua aplicação."
    },
    {
        "question": "Quais são os pré-requisitos para usar um plugin no StackSpot?",
        "answer": "Os pré-requisitos podem variar dependendo do plugin, mas geralmente incluem a instalação de dependências, criação de arquivos de configuração e pastas específicas."
    },
    {
        "question": "Como adicionar uma imagem na documentação?",
        "answer": "Para adicionar uma imagem na documentação, use a sintaxe `![Alt ou título da imagem](URL da imagem)` e forneça uma descrição clara e objetiva da imagem."
    },
    {
        "question": "Como criar uma lista ordenada em markdown?",
        "answer": "Para criar uma lista ordenada em markdown, coloque um número na frente de cada linha. Por exemplo:\n1. Primeiro item\n2. Segundo item\n3. Terceiro item"
    },
    {
        "question": "O que é Python?",
        "answer": "Python é uma linguagem de programação de alto nível, interpretada e de propósito geral. É conhecida por sua sintaxe clara e legível."
    },
    {
        "question": "Como instalar o Python?",
        "answer": "Você pode instalar o Python baixando o instalador do site oficial python.org e seguindo as instruções de instalação para o seu sistema operacional."
    },
    {
        "question": "O que são listas em Python?",
        "answer": "Listas são coleções ordenadas de itens que podem ser de diferentes tipos. Elas são definidas usando colchetes, por exemplo: [1, 2, 3, 'a', 'b', 'c']."
    },
    {
        "question": "Como criar uma função em Python?",
        "answer": "Para criar uma função em Python, use a palavra-chave 'def' seguida pelo nome da função e parênteses. Por exemplo:\ndef minha_funcao():\n    print('Olá, mundo!')"
    },
    {
        "question": "O que é PEP 8?",
        "answer": "PEP 8 é um guia de estilo para escrever código Python. Ele fornece convenções sobre como formatar o código para melhorar sua legibilidade."
    },
    {
        "question": "O que é Git?",
        "answer": "Git é um sistema de controle de versão distribuído, usado para rastrear mudanças no código-fonte durante o desenvolvimento de software."
    },
    {
        "question": "Como inicializar um repositório Git?",
        "answer": "Para inicializar um repositório Git, use o comando 'git init' no diretório do seu projeto."
    },
    {
        "question": "Como clonar um repositório Git?",
        "answer": "Para clonar um repositório Git, use o comando 'git clone <URL do repositório>'."
    },
    {
        "question": "O que é um commit no Git?",
        "answer": "Um commit no Git é uma operação que salva as mudanças no repositório. Cada commit tem uma mensagem que descreve as alterações feitas."
    },
    {
        "question": "Como criar um branch no Git?",
        "answer": "Para criar um branch no Git, use o comando 'git branch <nome-do-branch>'. Para mudar para o novo branch, use 'git checkout <nome-do-branch>'."
    }
]

In [ ]:
faq_collection = db_client.get_or_create_collection('faq')

faq_collection

In [ ]:
ids = []
meta = []
documents = []
for faq in faqs:
    ids.append(str(hashlib.md5(faq["question"].encode()).hexdigest()))
    meta.append({
        "datatype": "dict",
        "doc_type": "FAQ",
    })
    documents.append(f"""QUESTION: {faq["question"]}
ANSWER: {faq["answer"]}
""")

In [ ]:
faq_collection.add(ids, documents=documents, metadatas=meta)

In [ ]:
response = faq_collection.query(query_texts=input("VECTOR DATABASE: Qual sua dúvida hoje?"), n_results=2)

for i, resp in enumerate(response['documents'][0]):
    print(response['distances'][0][i])
    print(resp)

In [ ]:
prompt = input("RAG: Qual sua pergunta?")

docs = faq_collection.query(query_texts=prompt, n_results=3)["documents"][0]

In [ ]:
def generate_prompt(prompt, docs):
    messages_input = [
        {
            'role': 'system', 
            'content': f"You are a helpful assistant, answering user's questions"
        },
        {
            'role': 'system', 
            'content': f"Answer in portuguese"
        },
        {
            'role': 'system',
            'content': f'based in this document, answer the question: {"\n".join(docs)}'
        },
        {
            'role': 'user', 
            'content': prompt
        }
    ]
    return messages_input

In [ ]:
stream = llm_client.chat('llama3.1', generate_prompt(prompt, docs), stream=True)
print("\n".join(docs), end="\n"+("="*60)+"\n\n")
for chunk in stream:
    print(chunk["message"]["content"], end="")

In [ ]:
prompt = input("RAG Playground || O que deseja perguntar?")

docs = faq_collection.query(query_texts=prompt, n_results=1)["documents"][0]
print("\n".join(docs), end="\n"+("="*60)+"\n\n")

stream = llm_client.chat('gemma2:2b', generate_prompt(prompt, docs), stream=True)

for chunk in stream:
    print(chunk["message"]["content"], end="")